# Prefill/Decode Disaggregation

This notebook covers **P/D disaggregation**, one of llm-d's most powerful
optimization techniques. Instead of running both the prefill and decode phases
on the same GPU, you split them onto separate, specialized worker pools.

This is particularly effective for:
- Large models (70B+ parameters) with high throughput requirements
- Workloads with long input prompts and short output sequences
- Scenarios where you want to independently scale prefill and decode capacity

## Why Disaggregation?

LLM inference has two fundamentally different phases:

### Prefill Phase (Compute-Bound)
- Processes the entire input prompt in one forward pass
- Highly parallelizable across tokens - uses GPU compute efficiently
- Benefits from high FLOPS (compute throughput)
- Duration scales linearly with input sequence length (ISL)

### Decode Phase (Memory-Bandwidth-Bound)
- Generates one token at a time, autoregressively
- Each step reads the full KV-cache from GPU memory
- Bottlenecked by memory bandwidth, not compute
- Duration scales linearly with output sequence length (OSL)

### The Problem with Unified Serving

When both phases share the same GPU:
- A long prefill job **blocks** all decode jobs on that GPU, causing latency spikes
- GPUs are underutilized because prefill wants compute while decode wants bandwidth
- You cannot independently scale for different bottlenecks

### The Disaggregation Solution

Separate the phases onto different worker pools:
- **Prefill workers**: Optimized for compute (can use TP=1, smaller GPU count)
- **Decode workers**: Optimized for memory bandwidth (use TP=4 for more KV-cache)
- The KV-cache is transferred from prefill to decode workers after prefill completes

In [ ]:
# Set environment variables for P/D disaggregation
import os

os.environ["NAMESPACE"] = "llm-d"
os.environ["GUIDE_NAME"] = "pd-disaggregation"
os.environ["MODEL_NAME"] = "gpt-oss-120b"  # Example large model
os.environ["GUIDE_PATH"] = "llm-d-production-stack/guides/pd-disaggregation"

# Worker configuration
os.environ["PREFILL_REPLICAS"] = "8"   # More prefill workers for compute parallelism
os.environ["PREFILL_TP"] = "1"          # Tensor parallelism = 1 (1 GPU per prefill worker)
os.environ["DECODE_REPLICAS"] = "2"     # Fewer decode workers, each with more GPUs
os.environ["DECODE_TP"] = "4"           # Tensor parallelism = 4 (4 GPUs per decode worker)

print("P/D Disaggregation Configuration:")
print(f"  Model:             {os.environ['MODEL_NAME']}")
print(f"  Prefill workers:   {os.environ['PREFILL_REPLICAS']} x TP={os.environ['PREFILL_TP']}")
print(f"  Decode workers:    {os.environ['DECODE_REPLICAS']} x TP={os.environ['DECODE_TP']}")
print(f"  Total GPUs:        {int(os.environ['PREFILL_REPLICAS'])*int(os.environ['PREFILL_TP']) + int(os.environ['DECODE_REPLICAS'])*int(os.environ['DECODE_TP'])}")

In [ ]:
# Deploy prefill workers
# These are vLLM instances configured with --role prefill
# They only process the input prompt and generate the KV-cache

!kubectl apply -k $GUIDE_PATH/prefill-server/ -n $NAMESPACE

print("\nPrefill worker deployment applied.")
print(f"Waiting for {os.environ['PREFILL_REPLICAS']} prefill pods...")

!kubectl rollout status deployment/vllm-prefill -n $NAMESPACE --timeout=600s

print("Prefill workers are ready.")

In [ ]:
# Deploy decode workers
# These are vLLM instances configured with --role decode
# They receive the KV-cache from prefill workers and generate output tokens

!kubectl apply -k $GUIDE_PATH/decode-server/ -n $NAMESPACE

print("\nDecode worker deployment applied.")
print(f"Waiting for {os.environ['DECODE_REPLICAS']} decode pods...")

!kubectl rollout status deployment/vllm-decode -n $NAMESPACE --timeout=600s

print("Decode workers are ready.")

In [ ]:
# Deploy the disaggregated router configuration
# The EPP config for P/D disaggregation includes:
#   - Separate InferencePool definitions for prefill and decode
#   - A dispatch policy that routes initial requests to prefill
#     and follow-up decode to decode workers
#   - KV-cache transfer coordination

!kubectl apply -k $GUIDE_PATH/router/ -n $NAMESPACE

print("\nDisaggregated router deployment applied.")
print("Waiting for EPP pods...")

!kubectl rollout status deployment/epp -n $NAMESPACE --timeout=120s

print("Disaggregated router is ready.")

In [ ]:
# Verify all pods are running
print("=== All Pods ===")
!kubectl get pods -n $NAMESPACE -o wide -l 'app in (vllm-prefill,vllm-decode,epp)'

print("\n=== Prefill Pods ===")
!kubectl get pods -n $NAMESPACE -l app=vllm-prefill

print("\n=== Decode Pods ===")
!kubectl get pods -n $NAMESPACE -l app=vllm-decode

print("\n=== Router Pods ===")
!kubectl get pods -n $NAMESPACE -l app=epp

# Verify InferencePool configurations
print("\n=== InferencePools ===")
!kubectl get inferencepool -n $NAMESPACE

## Understanding the P/D Ratio

The ratio of prefill workers to decode workers is a critical tuning parameter.
The optimal ratio depends on your workload characteristics:

### Input-Heavy Workloads (High ISL, Low OSL)
- Example: Document summarization, RAG with large contexts
- ISL:OSL ratio might be 10:1 or higher
- **Use more prefill workers** (e.g., 8 prefill : 2 decode)
- Prefill is the bottleneck; decode finishes quickly

### Output-Heavy Workloads (Low ISL, High OSL)
- Example: Story generation, code completion with long outputs
- ISL:OSL ratio might be 1:5 or lower
- **Use more decode workers** (e.g., 2 prefill : 8 decode)
- Decode runs for a long time; prefill finishes quickly

### Balanced Workloads
- Example: Chatbots, general Q&A
- ISL:OSL ratio around 1:1 to 3:1
- **Start with equal resources** (e.g., 4 prefill : 4 decode)
- Monitor queue depths on each pool and adjust

### How to Measure

The EPP exposes per-pool queue depth metrics. If prefill queue depth is
consistently high while decode is idle, add more prefill workers (or vice versa).

```
epp_pool_queue_depth{pool="prefill"} 12.0    # High - needs more workers
epp_pool_queue_depth{pool="decode"}  1.0     # Low  - has spare capacity
```

In [ ]:
# Run a benchmark comparing disaggregated vs unified serving
import subprocess
import json
import time

GATEWAY_IP = subprocess.check_output(
    "kubectl get svc envoy-gateway -n llm-d -o jsonpath='{.status.loadBalancer.ingress[0].ip}'",
    shell=True
).decode().strip().strip("'")

print(f"Gateway endpoint: {GATEWAY_IP}:8080")
print()

# Simulate a long-input workload (high ISL) that benefits from disaggregation
long_context = "Kubernetes is a portable, extensible, open source platform for managing containerized workloads. " * 200

print(f"Input length: ~{len(long_context.split())} words")
print("Sending 5 requests with long input context...")
print()

!python3 -c "
import time, subprocess, json, statistics

context = 'Kubernetes is a portable, extensible, open source platform for managing containerized workloads. ' * 200

ttft_times = []
total_times = []

for i in range(5):
    payload = json.dumps({
        'model': 'gpt-oss-120b',
        'messages': [
            {'role': 'system', 'content': context},
            {'role': 'user', 'content': 'Summarize the above in one sentence.'},
        ],
        'max_tokens': 50,
        'stream': True,
    })

    start = time.time()
    proc = subprocess.Popen(
        ['curl', '-s', '-N', f'http://$GATEWAY_IP:8080/v1/chat/completions',
         '-H', 'Content-Type: application/json', '-d', payload],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE
    )

    first_token_time = None
    for line in proc.stdout:
        if first_token_time is None and b'data:' in line:
            first_token_time = time.time() - start

    total_time = time.time() - start
    if first_token_time:
        ttft_times.append(first_token_time)
        total_times.append(total_time)
        print(f'  Request {i+1}: TTFT={first_token_time:.3f}s, Total={total_time:.3f}s')
    else:
        print(f'  Request {i+1}: No response received')

if ttft_times:
    print()
    print(f'Median TTFT:  {statistics.median(ttft_times):.3f}s')
    print(f'Median Total: {statistics.median(total_times):.3f}s')
    print()
    print('With P/D disaggregation, TTFT should be more consistent because')
    print('prefill jobs do not block decode jobs on the same GPU.')
"

## Best Practices

### When to Use Disaggregation

Disaggregation adds complexity (KV-cache transfer, two pools to manage). Use it when:

- Your model is **large** (70B+ parameters) and you need high throughput
- Your workload has a **skewed ISL/OSL ratio** (very long inputs or very long outputs)
- You see **latency spikes** from prefill-decode interference in unified serving
- You want to **independently autoscale** prefill and decode capacity

### When NOT to Use Disaggregation

Stick with the optimized baseline when:
- Your model is small enough that prefill is fast (7B-13B parameters)
- Your traffic is low and a single replica handles it fine
- ISL and OSL are both short and similar in length

### ISL/OSL Considerations

| Metric | Prefill Impact | Decode Impact |
|--------|---------------|---------------|
| **High ISL** | Longer prefill time, more compute needed | More KV-cache to transfer |
| **High OSL** | No impact | Longer decode time, more memory bandwidth needed |
| **High concurrency** | Queue builds on prefill pool | Queue builds on decode pool |

### KV-Cache Transfer

The KV-cache transfer between prefill and decode workers uses high-speed
interconnect (RDMA/NVLink when available, TCP otherwise). Transfer latency
is typically 10-50ms for typical prompt lengths. This overhead is amortized
by the improved throughput from specialization.